# 05 - Complete Report, Power Analysis, and Data Audit

This notebook consolidates the final experimental metadata, direct measurements, derived pulse energies, confidence classifications, complete source-file inventory, and the outputs generated by the spatial, temporal, and spectral notebooks. It is intentionally conservative: a number is labeled as measured, derived, provisional, conditional, or unsupported according to what the apparatus actually resolves.

In [ ]:
from pathlib import Path
import json, sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image


def _locate_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for base in [start, *start.parents]:
        if (base / "experiment_metadata.json").is_file() and (base / "outputs").is_dir():
            return base.resolve()
        for child in list(base.glob("Laser_Characterization_Project*")) + list(base.glob("laser_characterization_project*")):
            if child.is_dir() and (child / "experiment_metadata.json").is_file() and (child / "outputs").is_dir():
                return child.resolve()
    raise FileNotFoundError("Could not locate the extracted laser-characterization project folder.")

ROOT = _locate_project_root()
OUT = ROOT / "outputs"
FIG = ROOT / "figures"
metadata = json.loads((ROOT / "experiment_metadata.json").read_text())
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)
print("Project:", ROOT)
print("Project version:", metadata.get("project_version"))

## Final measurement metadata

The values below are the experimenter-supplied final metadata. They supersede any provisional assumptions in earlier exploratory analysis.

In [ ]:
display(pd.json_normalize(metadata, sep="."))

## Optical power, pulse energy, and photon-number scale

For the pulsed modes, `E_pulse = P_average/f_rep`. Lower-bound powers remain lower-bound energies. The 65 ns value is retained as an approximate observed range. Peak power is not calculated as a measurement because the optical temporal profile is unresolved.

In [ ]:
power = pd.read_csv(OUT / "power_energy_summary.csv")
display(power)
display(Image(filename=str(FIG / "pulse_energy_v3.png")))

## Measurement confidence matrix

“High as electrical width” does not mean “high confidence optical pulse duration.” Confidence is attached to the precise quantity named in each row.

In [ ]:
confidence = pd.read_csv(OUT / "measurement_confidence_matrix.csv")
display(confidence)

## Complete source-file audit

In [ ]:
inventory = pd.read_csv(OUT / "source_data_inventory.csv")
summary = inventory.groupby(["domain", "extension"], dropna=False).agg(files=("source_id", "count"), uncompressed_bytes=("uncompressed_bytes", "sum")).reset_index()
display(summary)
print("Total source entries:", len(inventory))
display(inventory)

## Analysis-output catalog

In [ ]:
catalog = pd.read_csv(OUT / "analysis_output_catalog.csv")
display(catalog)

## Key figures from every domain

### Optical power summary

In [ ]:
display(Image(filename=str(FIG / "power_measurements_v3.png")))

## Final physical interpretation

1. The spatial data support sensor-plane beam centroids, selected-lobe dimensions, ellipticity, orientation, and repeatability. The fitted M2 values are provisional sensitivity results, not standards-grade source specifications.
2. The temporal data support a 200 kHz trigger-chain repetition rate and electrical response widths. They do not directly measure picosecond or 1 ns optical durations, and the nominal 65 ns pulse remains detector-convolved.
3. The narrow spectral features are instrument/sampling limited. The broadband picosecond spectra are well sampled and support spectral-bandwidth and conditional transform-limit calculations.
4. The visible Hg calibration anchors the 532 nm region much better than the 1064 nm region. Absolute centers near 1064 nm require a near-IR reference.
5. The reported average powers support pulse-energy bounds/ranges. Peak power remains unsupported without a reliable optical pulse duration and shape.
6. The approximately two-minute warm-up and missing dedicated camera background should be treated as reproducibility limitations, not silently absorbed into a fabricated uncertainty.